# Paper Citation Counts — C3 / C5 / C10 / C_all

Forward-citation counts received by each paper within four windows (**3, 5, 10, full**), from the MAG
paper citation graph. Primary key: `paper_id` (OpenAlex `W…`).

> Papers have no examiner/applicant tag (that is a patent-citation attribute), so only the total counts
> are produced here.

## Raw data (directory & structure)
```
/project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet/works/referenced_works/          # citing_mag <TAB> cited_mag (~1.8B edges)
/project/jevans/Dawoon/Science of Science/OpenAlex/cache/work_year_source_map.npz   # paper years
/project/jevans/Dawoon/Science of Science/OpenAlex/cache/paper_graph.npz           # cached (c_from,c_to,year,uni_mag) from paper_disruption
```
Reuses the cached graph built by `paper_disruption.ipynb` if present (else builds it — heavy 39 GB scan).

## Metric (formula)
$C_W(P)=\#\{c \text{ cites } P: 0 \le y_c - y_P \le W\}$; $C_{\mathrm{all}}$ counts all citers with $y_c \ge y_P$.

## Output
`/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_citation.parquet` — `paper_id, C_3, C_5, C_10, C_all`.

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa
ROOT = oa.BASE; OUT = oa.OUT
print('snapshot:', oa.ROOT)
OUT_FP = f'{OUT}/paper_citation.parquet'
WINSFX = [(3, '_3'), (5, '_5'), (10, '_10'), (-1, '_all')]
# PaperReferences.txt + paper_metadata.sqlite are replaced by works/referenced_works +
# works/works; oa.build_graph() writes the same c_from/c_to/year/uni_mag cache the rest of
# the pipeline loads.
# Everything below is fed by notebook/referenced_works_w_year.ipynb: it writes the per-work
# map (year + source) and the edge table with both years and both source ids, and
# oa.load_graph() / oa.load_csr() / oa.build_journal() read those instead of re-walking
# renli's tree. Build it once before running this notebook.
assert oa.have_consolidated(), (
    'run notebook/referenced_works_w_year.ipynb first — it builds the map and edge table')
oa.summary()

ROOT: C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science


## 1. Load (or build) the paper citation graph

In [2]:
%%time
c_from, c_to, year, uni_mag = oa.load_graph()   # builds the cache on first run
n = len(year)
print(f'{len(c_from):,} edges, {n:,} papers')

loaded cached graph: 1,540,740,924 edges, 189,903,764 papers
CPU times: total: 8.38 s
Wall time: 8.59 s


## 2. Count citers per window (CSR in-adjacency + year filter)

In [3]:
%%time
# forward citation counts per window, fully VECTORIZED over edges (citer = c_from, cited = c_to).
# window w: count citers q of paper P with 0 <= year[q] - year[P] <= w  (C_all = all with diff >= 0).
y = year.astype(np.int32)
diff = y[c_from] - y[c_to]                 # per-edge: citer_year - cited_year
del y; gc.collect()
valid = diff >= 0
res = {}
res['_all'] = np.bincount(c_to[valid], minlength=n).astype(np.int64)
for w, sfx in [(10, '_10'), (5, '_5'), (3, '_3')]:
    res[sfx] = np.bincount(c_to[valid & (diff <= w)], minlength=n).astype(np.int64)
del diff, valid; gc.collect()
print('citation counts computed (vectorized):', {k: int(v.sum()) for k, v in res.items()})

citation counts computed (vectorized): {'_all': 1529315256, '_10': 992050190, '_5': 611016168, '_3': 391932909}
CPU times: total: 1min 26s
Wall time: 1min 27s


## 3. Save + example

In [4]:
out = pd.DataFrame({'paper_id': np.char.add('W', uni_mag.astype(str))})
for _, s in WINSFX: out[f'C{s}'] = res[s]
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
for _, s in WINSFX: print(f'  C{s}: mean {out[f"C{s}"].mean():.2f}  >0 {(out[f"C{s}"]>0).mean()*100:.1f}%')
display(out.head(10))

WROTE C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science\notebook\paper\output\paper_citation.parquet  (189,903,764 rows, 5 cols)


  C_3: mean 2.06  >0 28.4%


  C_5: mean 3.22  >0 31.6%


  C_10: mean 5.22  >0 34.6%


  C_all: mean 8.05  >0 37.4%


,paper_id,C_3,C_5,C_10,C_all
0,W9,0,0,0,0
1,W15,0,0,0,0
2,W23,0,0,2,2
3,W58,0,0,0,0
4,W79,0,0,0,0
5,W87,0,0,0,0
6,W108,1,1,1,1
7,W125,1,1,1,1
8,W143,0,0,0,0
9,W147,0,0,0,0
